# Lab Assignment 7 — Tutorial Demo
## Using an RNN for a real application: **sentiment classification**

In Lab 6 you used an RNN as a **language model** — predicting the next word. But the lecture
(*Simple and LSTM Recurrent Neural Networks*, section 2, "Other uses of RNNs") lists several other
things an RNN can do. Today we build one of them end to end.

| Demo section | What we build |
|---|---|
| 1. Two shapes of RNN task | many-to-one vs many-to-many |
| 2. Preparing text for an RNN | vocabulary, integer encoding, padding |
| 3. The model | Embedding → SimpleRNN → Dense |
| 4. Training and evaluating | does it actually work? |
| 5. Where the SimpleRNN breaks | long sentences, and why |
| 6. What your assignment does instead | LSTM, and a different application |

> **Your assignment is deliberately NOT this task and NOT this architecture.** You will build a
> **POS tagger** using an **LSTM** — a different shape of problem and a different recurrent cell.

---
## 1. Two shapes of RNN task

An RNN reads a sequence one step at a time, keeping a **hidden state** that carries information
forward. What you do with those hidden states decides the task.

```
   many-to-one                              many-to-many
   (sentence classification)                (sequence tagging)

   h1   h2   h3   h4                        h1    h2    h3    h4
    │    │    │    │                         │     │     │     │
    └────┴────┴────┴──► ONE label            ▼     ▼     ▼     ▼
                        "positive"          DET   ADJ  NOUN  VERB
                                            one label PER WORD

   TODAY'S DEMO                             YOUR ASSIGNMENT
```

**Today (many-to-one).** The whole sentence gets **one** label: positive or negative. We read the
entire sentence, and the final hidden state becomes a summary of it — the lecture calls this the
**sentence encoding**. A classifier layer turns that summary into a label.

**Your assignment (many-to-many).** Every word gets its **own** label. The output is a sequence
the same length as the input.

> The lecture's exact words: *"RNNs can be used for sentence classification, e.g. sentiment
> classification"* — and *"how to compute sentence encoding? Basic way: use final hidden state."*
> That is precisely what we are about to build.

In [1]:
import numpy as np, keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

keras.utils.set_random_seed(42)      # so this demo reproduces exactly

MAX_WORDS = 5000     # vocabulary size
MAX_LEN   = 40       # every sentence padded/truncated to this many words

---
## 2. Preparing text for an RNN

A neural network needs numbers, and it needs them in a **rectangular** array. Real sentences are
different lengths, so there are three steps.

**Build a vocabulary.** Count every word, keep the most common `MAX_WORDS`, and give each one an
integer id. Two ids are reserved: `0` for padding and `1` for unknown words.

**Encode.** Replace each word by its id. `"the movie was great"` → `[4, 21, 8, 132]`.

**Pad.** Force every sequence to the same length by adding `0`s at the front (or cutting long
sentences short). This is what makes the batch rectangular.

In [2]:
labels_raw, texts = [], []
with open('mini_sentiment.tsv', encoding='utf-8') as f:
    next(f)                                     # skip the header row
    for line in f:
        label, sentence = line.rstrip('\n').split('\t')
        labels_raw.append(1 if label == 'pos' else 0)
        texts.append(sentence.split())

y = np.array(labels_raw)
print(f"{len(texts)} sentences   positive: {y.sum()}   negative: {len(y) - y.sum()}")
print(f"words per sentence: mean {np.mean([len(t) for t in texts]):.1f}, max {max(len(t) for t in texts)}")
print("\nexample:", ' '.join(texts[0][:14]), "...  ->", "positive" if y[0] else "negative")

10662 sentences   positive: 5331   negative: 5331
words per sentence: mean 21.0, max 59

example: think of it as gidget , only with muscles and a lot more smarts ...  -> positive


In [3]:
# --- build the vocabulary from word counts ---
counts = {}
for t in texts:
    for w in t:
        counts[w] = counts.get(w, 0) + 1

vocab = ['<pad>', '<unk>'] + [w for w, _ in sorted(counts.items(), key=lambda kv: -kv[1])[:MAX_WORDS]]
word_index = {w: i for i, w in enumerate(vocab)}

# --- encode, then pad ---
sequences = [[word_index.get(w, 1) for w in t] for t in texts]      # 1 = <unk>
X = pad_sequences(sequences, maxlen=MAX_LEN)                        # 0 = <pad>

print(f"vocabulary size: {len(vocab)}")
print(f"X shape: {X.shape}   (sentences x MAX_LEN)\n")
print("original :", ' '.join(texts[0][:10]), "...")
print("encoded  :", sequences[0][:10], "...")
print("padded   :", X[0][:18], "...")
print("\nNote the leading zeros: this sentence was shorter than", MAX_LEN, "words.")

vocabulary size: 5002
X shape: (10662, 40)   (sentences x MAX_LEN)

original : think of it as gidget , only with muscles and ...
encoded  : [247, 7, 12, 13, 1, 4, 65, 15, 1, 6] ...
padded   : [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 247   7] ...

Note the leading zeros: this sentence was shorter than 40 words.


---
## 3. The model

Three layers, and each one has a clear job.

| Layer | Job |
|---|---|
| `Embedding` | turns each word id into a dense vector the network can work with |
| `SimpleRNN` | reads the sentence left to right, updating a hidden state |
| `Dense(1, sigmoid)` | turns the final hidden state into one probability |

The key line is `SimpleRNN(32)`. By default a Keras RNN returns **only its last hidden state** —
a single 32-dimensional vector for the whole sentence. That vector *is* the **sentence encoding**
from the lecture. Everything the model knows about the sentence has to be in it.

$$h_t = \tanh\left(W_h h_{t-1} + W_x x_t + b\right)$$

Notice that $h_t$ depends on $h_{t-1}$, which depends on $h_{t-2}$, and so on. That chain is the
whole idea of a recurrent network — and, as we will see in section 5, also its weakness.

> Your assignment will need `return_sequences=True` instead, because tagging needs a hidden state
> at **every** step, not just the last one.

In [4]:
model = Sequential([
    Input(shape=(MAX_LEN,)),            # 40 word ids per sentence
    Embedding(len(vocab), 32),          # word id -> 32-dim vector
    SimpleRNN(32),                      # returns ONLY the final hidden state
    Dense(1, activation='sigmoid')      # final hidden state -> P(positive)
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 32)         │       160,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 162,177 (633.50 KB)

 Trainable params: 162,177 (633.50 KB)

 Non-trainable params: 0 (0.00 B)

---
## 4. Training and evaluating

We hold out the last 2,662 sentences as a test set the model never sees during training.
Binary crossentropy is the loss because this is a two-class problem.

In [5]:
X_train, y_train = X[:8000], y[:8000]
X_test,  y_test  = X[8000:], y[8000:]

history = model.fit(X_train, y_train, epochs=4, batch_size=64,
                    validation_split=0.2, verbose=0)

h = history.history
print(f"{'epoch':>6}{'loss':>9}{'accuracy':>10}{'val_loss':>10}{'val_accuracy':>14}")
for e in range(len(h['loss'])):
    print(f"{e+1:>6}{h['loss'][e]:>9.4f}{h['accuracy'][e]:>10.4f}"
          f"{h['val_loss'][e]:>10.4f}{h['val_accuracy'][e]:>14.4f}")

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTEST ACCURACY: {test_acc:.3f}   (random guessing would be 0.500)")

 epoch     loss  accuracy  val_loss  val_accuracy
     1   0.6899    0.5348    0.6823        0.5769
     2   0.5149    0.7855    0.6459        0.6438
     3   0.2003    0.9420    0.7482        0.6481
     4   0.0754    0.9852    0.8796        0.6300



TEST ACCURACY: 0.639   (random guessing would be 0.500)


In [6]:
# --- try it on our own sentences ---
def predict_sentiment(sentence):
    seq = [[word_index.get(w, 1) for w in sentence.lower().split()]]
    p = float(model.predict(pad_sequences(seq, maxlen=MAX_LEN), verbose=0)[0, 0])
    return p

print(f"{'SENTENCE':<52}{'P(positive)':>12}   VERDICT")
print("-" * 80)
for s in ["a beautiful and moving film",
          "the plot was dull and the acting was terrible",
          "one of the best movies of the year",
          "i wasted two hours of my life",
          "the film is not without charm"]:
    p = predict_sentiment(s)
    print(f"{s:<52}{p:>12.3f}   {'positive' if p > .5 else 'negative'}")

SENTENCE                                             P(positive)   VERDICT
--------------------------------------------------------------------------------
a beautiful and moving film                                0.974   positive
the plot was dull and the acting was terrible              0.008   negative
one of the best movies of the year                         0.985   positive


i wasted two hours of my life                              0.269   negative
the film is not without charm                              0.309   negative


---
## 5. Where the SimpleRNN breaks

About 64% is clearly better than guessing, but it is not good.

Two things are worth noticing in the training table above. Training accuracy climbs to 0.98 while
validation accuracy stalls near 0.64 — the model is **overfitting**, memorising the training
sentences. And the test accuracy is low in absolute terms.

But the more interesting question is *where* it fails. Let us split the test set by sentence
length and score each group separately.

We split the test set by **sentence length** and score each group separately.

In [7]:
raw_len = np.array([len(t) for t in texts])[8000:]
pred = (model.predict(X_test, verbose=0).ravel() > 0.5).astype(int)

short = raw_len <= 15
long_ = raw_len >= 28

print(f"{'GROUP':<26}{'n':>6}{'accuracy':>11}")
print("-" * 44)
print(f"{'SHORT (<= 15 words)':<26}{short.sum():>6}{(pred[short] == y_test[short]).mean():>11.3f}")
print(f"{'ALL':<26}{len(y_test):>6}{(pred == y_test).mean():>11.3f}")
print(f"{'LONG  (>= 28 words)':<26}{long_.sum():>6}{(pred[long_] == y_test[long_]).mean():>11.3f}")
print("\nAccuracy falls as sentences get longer.")

GROUP                          n   accuracy
--------------------------------------------
SHORT (<= 15 words)          808      0.705
ALL                         2662      0.639
LONG  (>= 28 words)          680      0.584

Accuracy falls as sentences get longer.


### Why longer is harder — the vanishing gradient

The lecture's section 3 explains it. To learn from a word near the **start** of a long sentence,
the training signal has to travel back through every step in between:

$$\frac{\partial \text{loss}}{\partial h_1} = \frac{\partial \text{loss}}{\partial h_T}
\cdot \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

That product has one term per step. If the terms are smaller than 1, the whole thing shrinks
**exponentially** with distance. By the time it reaches the early words, the gradient is
effectively zero, so the model never learns long-range dependencies.

Concretely, in *"the first half is slow, but the ending is absolutely wonderful"* the verdict
depends on words the SimpleRNN has largely forgotten by the time it finishes reading.

The lecture's conclusion slide puts it plainly: **"LSTMs are powerful."** An LSTM adds a separate
**cell state** with gates that control what to keep and what to forget, giving the gradient a path
that does not shrink at every step.

---
## 6. What your assignment does

Your assignment picks up exactly where this demo stops.

### Task 1 — test the claim I just made

Section 5 left you with an accusation, not a proof: *the SimpleRNN fails on long sentences, and an
LSTM would fix it.* Task 1 makes you check.

You rebuild **this same classifier**, on **this same data**, changing exactly one line —
`SimpleRNN(32)` becomes `LSTM(32)` — and score it the same three ways. Then you can answer:

- Does the LSTM actually do better?
- And is the gain **largest on the long sentences**, where the vanishing gradient does the most
  damage?

Because only one thing changes, whatever difference you measure is caused by the recurrent cell
and nothing else. That is what makes it a fair test rather than a guess.

### Tasks 2 to 5 — a different kind of problem

Then you take the LSTM somewhere new: **part-of-speech tagging**, where every word gets its own
label.

| | This demo | Tasks 2–5 |
|---|---|---|
| Application | sentiment classification | **POS tagging** |
| Shape | many-to-one (one label per sentence) | **many-to-many** (one label per word) |
| Cell | `SimpleRNN` | **`LSTM`**, then a **`Bidirectional`** LSTM |
| Output layer | `Dense(1)` on the last state | `TimeDistributed(Dense(n_tags))` on **every** state |
| Data | movie review sentences | the **Penn Treebank**, tagged |

You met POS tagging before, in **Lab 3, with a Hidden Markov Model**. Now you will do the same job
with a recurrent network and can compare the two.

### What carries over, and what does not

Two things from today transfer directly: **integer encoding with padding**, and the idea that the
hidden state is a running summary of what has been read.

Two things change. You will need `return_sequences=True`, because tagging needs a hidden state at
**every** step rather than only the last. And you must be careful **not to count the padding** when
you measure accuracy — that one catches almost everybody.

> **No LSTM code here — that is your assignment.**